In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.decomposition import PCA
from IPython.display import clear_output
import time

np.set_printoptions(precision=3, suppress=True, floatmode = 'fixed')
pd.options.display.float_format = '{:.3f}'.format

# Data

## Load Data

Dataset: players_22.

Trata-se de uma tabela de dados contendo atributos e estatísticas de jogadores do videogame FIFA 22.

Inclui mais de 19.000 jogadores e estatísticas como potencial, salários, valores e mais de 30 atributos de habilidade.

In [ ]:
players = pd.read_csv('data/players_22.csv', low_memory=False)

In [ ]:
players.shape

In [ ]:
players.sample(5)

In [ ]:
players.info()

In [ ]:
players.isna().sum().sort_values(ascending=False)

In [ ]:
na_series = players.isna().sum()
na_series

In [ ]:
threshold = len(players) * 0.05

In [ ]:
cols_to_drop = na_series[na_series > threshold]
cols_to_drop

In [ ]:
players = players.drop(columns = cols_to_drop.index)

In [ ]:
players.info()

In [ ]:
num_cols = players.select_dtypes(include='number').columns

In [ ]:
players[num_cols] = players[num_cols].astype('float64')

In [ ]:
players.info()

In [ ]:
# Só iremos trabalhar com essas 5 variáves.

columns = ['overall', 'potential', 'wage_eur', 'value_eur', 'age']

In [ ]:
# Drop missing values

players = players.dropna(subset = columns, axis = 0, ignore_index=True)

In [ ]:
players.isna().values.sum(axis=0).sum()

## Create Dataframe

In [ ]:
df = players[columns].copy()

In [ ]:
df.sample(10)

In [ ]:
df.describe()

In [ ]:
df.shape

## Normalize

In [ ]:
df.min()

In [ ]:
df.max()

In [ ]:
df_scaled = ((df - df.min()) / (df.max() - df.min()))

In [ ]:
df_scaled.describe()

## Boxplot

In [ ]:
fig = px.box(df_scaled)
fig.show()

Nenhum outlier será removido.

# Convert to numpy.array

In [ ]:
X = df_scaled.to_numpy()

# K-Means Code

In [ ]:
def get_labels(X, centroids, X_norm = None):

    # X = (n, d)
    # c = (k, d)

    # (n, 1, d) - (1, k, d) = (n, k, d) → sum(axis=d) → (n, k)
    # distances = (n, k)

    # labels = argmin(distances, axis=k) → (n, 1)

    if X_norm is None:
        X_norm = np.sum(X**2, axis=1, keepdims=True) # (n,1)
    
    C_norm = np.sum(centroids**2, axis=1) # (k,)

    distances = X_norm + C_norm - 2 * (X @ centroids.T) # (n, k)

    labels = np.argmin(distances, axis=1)
   
    return labels

In [ ]:
def update_centroids(X, labels, centroids):
    
    k, d = centroids.shape

    new_centroids = np.zeros((k,d))
    counts = np.zeros(k)

    np.add.at(new_centroids, labels, X)

    np.add.at(counts, labels, 1)

    mask = counts > 0

    new_centroids[mask] /= counts[mask].reshape(-1,1)
    new_centroids[~mask] = centroids[~mask]
    
    return new_centroids

In [ ]:
def get_wcss(X, labels, centroids):

    diff = X - centroids[labels]   
    return np.sum((diff ** 2))

# Elbow Test

In [ ]:
n_clusters = [2,3,4,5,6,7,8,9,10,11] # número de centroides

In [ ]:
start_time = time.time()
X_norm = np.sum(X**2, axis=1, keepdims=True)
wcss = []

for k in n_clusters:
    idx = np.random.choice(len(X), k, replace=False)
    centroids = np.zeros((k, X.shape[1]))
    new_centroids = X[idx]
    n_iter = 0
    max_iter = 100
    
    while n_iter < max_iter and not np.allclose(centroids, new_centroids, atol=1e-8):
        centroids = new_centroids
        labels = get_labels(X, centroids, X_norm)
        new_centroids = update_centroids(X, labels, centroids)
        n_iter+=1
    
    wcss.append(get_wcss(X, labels, new_centroids))

end_time = time.time()
total_sec = end_time - start_time

minutes = total_sec // 60
seconds = total_sec % 60

print(f"Tempo decorrido: {minutes} min {seconds:.2f} seg.")

In [ ]:
fig, ax = plt.subplots()
ax.plot(n_clusters, wcss)
plt.show()

# Choose k

Escolha o k conforme o teste do cotovelo

In [ ]:
k_optim = 5

# Plot Function

In [ ]:
pca = PCA(n_components = 2)
data_2d = pca.fit_transform(X)

In [ ]:
def plot_clusters(data_2d, labels, centroids, iteration, pca):
    centroids_2d = pca.transform(centroids)
    
    index, counts = np.unique(labels, return_counts=True)
    codes = dict(zip(index,counts))
    
    clear_output(wait=True)
    
    fig, ax = plt.subplots()
    scatter = ax.scatter(x = data_2d[:,0], y = data_2d[:,1], c = labels)
    ax.scatter(x = centroids_2d[:,0], y = centroids_2d[:,1], marker = 'X', s=50, c = 'red')
   
    handles = scatter.legend_elements(prop='colors')[0]
    ax.legend(handles=handles, labels=codes.values())
    
    plt.title(f'Iteração {iteration}')
    plt.show()

# Iteration

In [ ]:
n_iter = 0
max_iter = 100

idx = np.random.choice(len(X), k_optim, replace=False)
centroids = np.zeros((k_optim, X.shape[1]))
new_centroids = X[idx]

In [ ]:
while n_iter < max_iter and not np.allclose(centroids, new_centroids, atol=1e-6):
    centroids = new_centroids
    labels = get_labels(X, centroids)
    new_centroids = update_centroids(X, labels, centroids)
    n_iter+=1
    plot_clusters(data_2d, labels, new_centroids, n_iter, pca)
    print(f"Centroids:\n{centroids}")

# End